In [22]:
import json
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split,RandomizedSearchCV
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score
from sklearn.svm import SVC
from sklearn.pipeline import Pipeline



In [2]:
# 1. Load and prepare data
with open('train_part1.json', 'r') as f:
    data = json.load(f)

X = []
y = []
for record in data:
    features = record['image_embedding'] + record['text_embedding']
    X.append(features)
    y.append(record['label'])

X = np.array(X)
y = np.array(y)

In [3]:
# 2. Split data
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [4]:
# 3. Standardize features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)

In [6]:
# 4. Apply PCA to reduce dimensions
pca = PCA(n_components=.9)  # Reduce from 1024 to 100
X_train_pca = pca.fit_transform(X_train_scaled)
X_val_pca = pca.transform(X_val_scaled)

print(f"Explained variance: {pca.explained_variance_ratio_.sum():.3f}")

Explained variance: 0.901


In [20]:
param_grid = {
    'n_estimators': [100, 200, 300, 400, 500],      # number of trees
    'max_depth': [None, 10, 20, 30, 40],            # max depth of each tree
    'min_samples_split': [2, 5, 10],                # minimum samples required to split a node
    'min_samples_leaf': [1, 2, 4],                  # minimum samples required at a leaf
    'max_features': ['sqrt', 'log2', None],         # number of features to consider for split
    'bootstrap': [True, False],                     # use bootstrap samples or not
    'class_weight': [None, 'balanced']             # handle class imbalance
}


In [23]:
model = RandomForestClassifier()

In [24]:
random_search = RandomizedSearchCV(
    estimator=model,
    param_distributions=param_grid,
    n_iter=50,                  # number of random combinations
    scoring='f1_macro',         # for multi-class F1-macro
    n_jobs=-1,
    cv=5,
    verbose=2,
    random_state=42,
    error_score='raise'
)

random_search.fit(X_train, y_train)

print("Best parameters:", random_search.best_params_)
print("Best F1-macro score:", random_search.best_score_)


Fitting 5 folds for each of 50 candidates, totalling 250 fits
Best parameters: {'n_estimators': 400, 'min_samples_split': 2, 'min_samples_leaf': 1, 'max_features': None, 'max_depth': 10, 'class_weight': None, 'bootstrap': False}
Best F1-macro score: 0.5814113318393835


In [17]:
y_pred = model.predict(X_val_pca)
f1_macro = f1_score(y_val, y_pred, average='macro')
print(f"Validation F1 Macro Score: {f1_macro:.4f}")

Validation F1 Macro Score: 0.4887


In [18]:
# 7. Predict on test set
with open('test.json', 'r') as f:
    test_data = json.load(f)

X_test = np.array([
    record['image_embedding'] + record['text_embedding']
    for record in test_data
])
test_ids = [record['id'] for record in test_data]

X_test_scaled = scaler.transform(X_test)
X_test_pca = pca.transform(X_test_scaled)
predictions = model.predict(X_test_pca)



In [19]:
# 8. Create submission
submission = pd.DataFrame({
    'row_id': test_ids,
    'target': predictions
})
submission.to_csv('rf_baseline_pca.csv', index=False)